In [1]:
import numpy as np

In [2]:
def calculate_entropy(y):
    # Contiamo quante volte appare ogni classe
    counts = np.bincount(y)
    probabilities = counts / len(y)
    # Calcolo entropia: -somma(p * log2(p))
    return -np.sum([p * np.log2(p) for p in probabilities if p > 0])

In [3]:
class DecisionStump:
    def __init__(self):
        self.feature_idx = None
        self.threshold = None
        self.left_value = None
        self.right_value = None

    def fit(self, X, y):
        n_samples, n_features = X.shape
        best_gain = -1
        
        # L'entropia iniziale dei dati senza split
        parent_entropy = calculate_entropy(y)

        for f_idx in range(n_features):
            # Proviamo ogni valore presente nei dati come possibile soglia
            thresholds = np.unique(X[:, f_idx])
            for threshold in thresholds:
                # Splittiamo i dati
                left_idxs = np.where(X[:, f_idx] <= threshold)[0]
                right_idxs = np.where(X[:, f_idx] > threshold)[0]

                if len(left_idxs) == 0 or len(right_idxs) == 0:
                    continue

                # Calcoliamo l'entropia pesata dei due nuovi rami
                n = len(y)
                n_l, n_r = len(left_idxs), len(right_idxs)
                e_l, e_r = calculate_entropy(y[left_idxs]), calculate_entropy(y[right_idxs])
                child_entropy = (n_l/n) * e_l + (n_r/n) * e_r

                # Guadagno d'informazione
                ig = parent_entropy - child_entropy

                if ig > best_gain:
                    best_gain = ig
                    self.feature_idx = f_idx
                    self.threshold = threshold
                    # Salviamo la classe più frequente per ogni ramo
                    self.left_value = np.bincount(y[left_idxs]).argmax()
                    self.right_value = np.bincount(y[right_idxs]).argmax()

    def predict(self, X):
        return np.where(X[:, self.feature_idx] <= self.threshold, 
                        self.left_value, self.right_value)

In [4]:
X = np.array([1, 3, 5, 7, 9, 11, 13, 15, 17, 19]).reshape(-1, 1)
y = np.array([0, 0, 0, 0, 0, 1, 1, 1, 1, 1])

stump = DecisionStump()
stump.fit(X, y)

print(f"Migliore soglia trovata: {stump.threshold}")
# Dovrebbe stampare 9 (perché split <= 9 va a sinistra, > 9 va a destra)

Migliore soglia trovata: 9


In [5]:
class Node:
    def __init__(self, feature=None, threshold=None, left=None, right=None, value=None):
        self.feature = feature     # Indice della feature usata per lo split
        self.threshold = threshold # Valore di soglia
        self.left = left           # Ramo sinistro
        self.right = right         # Ramo destro
        self.value = value         # Se è una foglia, contiene la classe finale

class DecisionTree:
    def __init__(self, max_depth=10, n_features=None):
        self.max_depth = max_depth
        self.n_features = n_features # Per Random Forest: quante feature usare a ogni split
        self.root = None

    def fit(self, X, y):
        # Se non specificato, usiamo tutte le feature
        self.n_features = X.shape[1] if not self.n_features else min(X.shape[1], self.n_features)
        self.root = self._grow_tree(X, y)

    def _grow_tree(self, X, y, depth=0):
        n_samples, n_feats = X.shape
        n_labels = len(np.unique(y))

        # Criteri di arresto: profondità raggiunta o classe pura
        if depth >= self.max_depth or n_labels == 1:
            return Node(value=np.bincount(y).argmax())

        # Selezione casuale delle feature (Cuore della Random Forest)
        feat_idxs = np.random.choice(n_feats, self.n_features, replace=False)

        # Trova il miglior split (usando la logica dello stump)
        best_feat, best_thresh = self._best_split(X, y, feat_idxs)

        # Crea i figli recursivamente
        left_idxs = X[:, best_feat] <= best_thresh
        right_idxs = X[:, best_feat] > best_thresh
        left = self._grow_tree(X[left_idxs], y[left_idxs], depth + 1)
        right = self._grow_tree(X[right_idxs], y[right_idxs], depth + 1)
        return Node(best_feat, best_thresh, left, right)

    def _best_split(self, X, y, feat_idxs):
        # ... qui va la logica del DecisionStump che abbiamo scritto prima ...
        # (Cerca lo split che massimizza l'Information Gain)
        pass # Per brevità saltiamo i calcoli matematici già visti

    def predict(self, X):
        return np.array([self._traverse_tree(x, self.root) for x in X])

    def _traverse_tree(self, x, node):
        if node.value is not None: return node.value
        if x[node.feature] <= node.threshold:
            return self._traverse_tree(x, node.left)
        return self._traverse_tree(x, node.right)

In [6]:
class RandomForestScratch:
    def __init__(self, n_trees=5, max_depth=10, n_features=None):
        self.n_trees = n_trees
        self.max_depth = max_depth
        self.n_features = n_features
        self.trees = []

    def fit(self, X, y):
        self.trees = []
        for _ in range(self.n_trees):
            # 1. Crea un albero
            tree = DecisionTree(max_depth=self.max_depth, n_features=self.n_features)
            
            # 2. Bootstrap: campiona i dati con ripetizione
            n_samples = X.shape[0]
            idxs = np.random.choice(n_samples, n_samples, replace=True)
            X_sample, y_sample = X[idxs], y[idxs]
            
            # 3. Addestra l'albero sul campione
            tree.fit(X_sample, y_sample)
            self.trees.append(tree)

    def predict(self, X):
        # Ogni albero vota
        tree_preds = np.array([tree.predict(X) for tree in self.trees])
        # tree_preds ha forma (n_alberi, n_campioni). Trasponiamo.
        tree_preds = np.swapaxes(tree_preds, 0, 1)
        # Maggioranza dei voti per ogni riga
        return np.array([np.bincount(p).argmax() for p in tree_preds])

In [7]:
# Aggiungiamo la funzione di utilità per l'entropia fuori dalla classe
def calculate_entropy(y):
    counts = np.bincount(y)
    probs = counts / len(y)
    return -np.sum([p * np.log2(p) for p in probs if p > 0])

# Integriamo la logica mancante nel tuo DecisionTree
def _best_split_logic(self, X, y, feat_idxs):
    best_gain = -1
    split_idx, split_thresh = None, None
    parent_entropy = calculate_entropy(y)

    for feat_idx in feat_idxs:
        X_column = X[:, feat_idx]
        thresholds = np.unique(X_column)
        for threshold in thresholds:
            # Split
            left_idxs = X_column <= threshold
            right_idxs = X_column > threshold
            
            if len(y[left_idxs]) == 0 or len(y[right_idxs]) == 0:
                continue
            
            # Information Gain
            n = len(y)
            n_l, n_r = len(y[left_idxs]), len(y[right_idxs])
            e_l, e_r = calculate_entropy(y[left_idxs]), calculate_entropy(y[right_idxs])
            child_entropy = (n_l/n) * e_l + (n_r/n) * e_r
            ig = parent_entropy - child_entropy

            if ig > best_gain:
                best_gain = ig
                split_idx = feat_idx
                split_thresh = threshold
                
    return split_idx, split_thresh

# Monkey-patch per iniettare la logica nel tuo codice esistente
DecisionTree._best_split = _best_split_logic

In [8]:
# --- CELLA DI TEST ---

# 1. Creazione dati giocattolo (X: 1-20, y: soglia a 10)
X_toy = np.array([1, 3, 5, 7, 9, 11, 13, 15, 17, 19]).reshape(-1, 1)
y_toy = np.array([0, 0, 0, 0, 0, 1, 1, 1, 1, 1])

# 2. Inizializzazione Random Forest
# Usiamo 3 alberi piccoli per i dati giocattolo
rf = RandomForestScratch(n_trees=3, max_depth=3)

# 3. Addestramento
print("Addestramento della Random Forest in corso...")
rf.fit(X_toy, y_toy)

# 4. Predizione e Risultati
predictions = rf.predict(X_toy)

print("\n--- RISULTATI TOY DATA ---")
print(f"Input:       {X_toy.flatten()}")
print(f"Reali:       {y_toy}")
print(f"Predizioni:  {predictions}")

# Verifica Accuratezza
accuracy = np.mean(predictions == y_toy)
print(f"\nAccuratezza Finale: {accuracy * 100}%")

# Vediamo cosa dicono i singoli alberi (per capire il voto di maggioranza)
print("\nVoti dei singoli alberi per ogni campione:")
for i, tree in enumerate(rf.trees):
    print(f"Albero {i+1}: {tree.predict(X_toy)}")

Addestramento della Random Forest in corso...

--- RISULTATI TOY DATA ---
Input:       [ 1  3  5  7  9 11 13 15 17 19]
Reali:       [0 0 0 0 0 1 1 1 1 1]
Predizioni:  [0 0 0 0 0 1 1 1 1 1]

Accuratezza Finale: 100.0%

Voti dei singoli alberi per ogni campione:
Albero 1: [0 0 0 0 0 1 1 1 1 1]
Albero 2: [0 0 0 0 0 1 1 1 1 1]
Albero 3: [0 0 0 0 0 1 1 1 1 1]


In [9]:
class DecisionStump:
    def __init__(self):
        self.feature_idx = None
        self.threshold = None
        self.polarity = 1
        self.alpha = None # Peso dello stump nella decisione finale

    def predict(self, X):
        n_samples = X.shape[0]
        X_column = X[:, self.feature_idx]
        predictions = np.ones(n_samples)
        if self.polarity == 1:
            predictions[X_column <= self.threshold] = 0
        else:
            predictions[X_column <= self.threshold] = 1
        return predictions

class AdaBoostScratch:
    def __init__(self, n_stumps=5):
        self.n_stumps = n_stumps
        self.stumps = []

    def fit(self, X, y):
        n_samples, n_features = X.shape
        # Inizializziamo i pesi: all'inizio tutti i dati sono ugualmente importanti
        w = np.full(n_samples, (1 / n_samples))

        for _ in range(self.n_stumps):
            stump = DecisionStump()
            min_error = float('inf')

            # Cerchiamo il miglior split considerando i pesi w
            for f_idx in range(n_features):
                X_column = X[:, f_idx]
                thresholds = np.unique(X_column)
                for threshold in thresholds:
                    for polarity in [1, -1]:
                        # Generiamo predizioni
                        p = np.ones(n_samples)
                        if polarity == 1:
                            p[X_column <= threshold] = 0
                        else:
                            p[X_column <= threshold] = 1
                        
                        # ERRORE PESATO: somma dei pesi dei campioni sbagliati
                        error = sum(w[y != p])

                        if error < min_error:
                            min_error = error
                            stump.feature_idx = f_idx
                            stump.threshold = threshold
                            stump.polarity = polarity

            # Calcolo di Alpha (quanto è affidabile questo stump)
            # eps serve a non dividere per zero se l'errore è 0
            eps = 1e-10
            stump.alpha = 0.5 * np.log((1.0 - min_error + eps) / (min_error + eps))

            # Aggiornamento dei pesi w
            # I campioni sbagliati avranno un peso maggiore nel prossimo round
            predictions = stump.predict(X)
            # Formula: w = w * exp(-alpha * y_reale * y_pred)
            # Nota: y deve essere -1 o 1 per la matematica di AdaBoost
            y_math = np.where(y == 0, -1, 1)
            p_math = np.where(predictions == 0, -1, 1)
            w *= np.exp(-stump.alpha * y_math * p_math)
            
            # Normalizziamo w affinché la somma sia 1
            w /= np.sum(w)

            self.stumps.append(stump)

    def predict(self, X):
        # Ogni stump vota moltiplicando la sua predizione per il suo Alpha
        stump_preds = []
        for stump in self.stumps:
            p = stump.predict(X)
            p_math = np.where(p == 0, -1, 1)
            stump_preds.append(stump.alpha * p_math)
        
        # Sommiamo i voti pesati
        y_final = np.sum(stump_preds, axis=0)
        # Se la somma è > 0 allora classe 1, altrimenti 0
        return np.where(y_final >= 0, 1, 0)

In [10]:
# Dati Toy
X = np.array([1, 3, 5, 7, 9, 11, 13, 15, 17, 19]).reshape(-1, 1)
y = np.array([0, 0, 0, 0, 0, 1, 1, 1, 1, 1])

# Creiamo il modello AdaBoost
aboost = AdaBoostScratch(n_stumps=3)
aboost.fit(X, y)
preds = aboost.predict(X)

print("--- TEST ADABOOST ---")
print(f"Input:      {X.flatten()}")
print(f"Predizioni: {preds}")
print(f"Corrette:   {y}")

# Analisi della forza degli stump
for i, s in enumerate(aboost.stumps):
    print(f"Stump {i+1}: Soglia={s.threshold}, Peso (Alpha)={s.alpha:.4f}")

--- TEST ADABOOST ---
Input:      [ 1  3  5  7  9 11 13 15 17 19]
Predizioni: [0 0 0 0 0 1 1 1 1 1]
Corrette:   [0 0 0 0 0 1 1 1 1 1]
Stump 1: Soglia=9, Peso (Alpha)=11.5129
Stump 2: Soglia=9, Peso (Alpha)=11.5129
Stump 3: Soglia=9, Peso (Alpha)=11.5129


In [11]:
class SimpleRegressionTree:
    """Un albero di regressione ultra-semplice (profondità 1) per i residui"""
    def __init__(self):
        self.feature_idx = None
        self.threshold = None
        self.left_val = None
        self.right_val = None

    def fit(self, X, residuals):
        n_samples, n_features = X.shape
        min_mse = float('inf')

        for f_idx in range(n_features):
            thresholds = np.unique(X[:, f_idx])
            for thresh in thresholds:
                left_idxs = X[:, f_idx] <= thresh
                right_idxs = X[:, f_idx] > thresh
                
                if not any(left_idxs) or not any(right_idxs): continue

                # Valore predetto = media dei residui in quel ramo
                l_val = np.mean(residuals[left_idxs])
                r_val = np.mean(residuals[right_idxs])

                # Calcolo Errore Quadratico Medio (MSE)
                mse = np.sum((residuals[left_idxs] - l_val)**2) + \
                      np.sum((residuals[right_idxs] - r_val)**2)

                if mse < min_mse:
                    min_mse = mse
                    self.feature_idx = f_idx
                    self.threshold = thresh
                    self.left_val = l_val
                    self.right_val = r_val

    def predict(self, X):
        return np.where(X[:, self.feature_idx] <= self.threshold, 
                        self.left_val, self.right_val)

class GradientBoostingScratch:
    def __init__(self, n_estimators=5, lr=0.1):
        self.n_estimators = n_estimators
        self.lr = lr
        self.trees = []
        self.init_pred = None

    def fit(self, X, y):
        # 1. Predizione iniziale (media)
        self.init_pred = np.mean(y)
        current_preds = np.full(len(y), self.init_pred)

        for _ in range(self.n_estimators):
            # 2. Calcolo residui
            residuals = y - current_preds

            # 3. Addestramento albero sui residui
            tree = SimpleRegressionTree()
            tree.fit(X, residuals)
            
            # 4. Aggiornamento predizioni
            current_preds += self.lr * tree.predict(X)
            self.trees.append(tree)

    def predict(self, X):
        y_pred = np.full(X.shape[0], self.init_pred)
        for tree in self.trees:
            y_pred += self.lr * tree.predict(X)
        # Per classificazione binaria, usiamo 0.5 come soglia
        return np.where(y_pred >= 0.5, 1, 0)

In [12]:
X = np.array([1, 3, 5, 7, 9, 11, 13, 15, 17, 19]).reshape(-1, 1)
y = np.array([0, 0, 0, 0, 0, 1, 1, 1, 1, 1])

gb = GradientBoostingScratch(n_estimators=10, lr=0.1)
gb.fit(X, y)
preds = gb.predict(X)

print(f"Predizioni finali: {preds}")
print(f"Loss finale: {np.mean((y - preds)**2):.4f}")

Predizioni finali: [0 0 0 0 0 1 1 1 1 1]
Loss finale: 0.0000


In [13]:
class XGBoostTree:
    """Singolo albero che ottimizza il Similarity Score"""
    def __init__(self, lambd=1.0):
        self.lambd = lambd # Regolarizzazione L2
        self.feature_idx = None
        self.threshold = None
        self.left_val = None
        self.right_val = None

    def fit(self, X, g, h):
        n_samples, n_features = X.shape
        max_gain = 0
        # Similarity score del nodo radice
        root_similarity = (np.sum(g)**2) / (np.sum(h) + self.lambd)

        for f_idx in range(n_features):
            X_column = X[:, f_idx]
            thresholds = np.unique(X_column)
            for thresh in thresholds:
                left_mask = X_column <= thresh
                right_mask = X_column > thresh
                
                if not any(left_mask) or not any(right_mask): continue

                # Similarity score per i rami figli
                sim_l = (np.sum(g[left_mask])**2) / (np.sum(h[left_mask]) + self.lambd)
                sim_r = (np.sum(g[right_mask])**2) / (np.sum(h[right_mask]) + self.lambd)
                
                # Calcolo del Guadagno (Gain)
                gain = sim_l + sim_r - root_similarity

                if gain > max_gain:
                    max_gain = gain
                    self.feature_idx = f_idx
                    self.threshold = thresh
                    # Calcolo dei pesi ottimali (output delle foglie)
                    self.left_val = -np.sum(g[left_mask]) / (np.sum(h[left_mask]) + self.lambd)
                    self.right_val = -np.sum(g[right_mask]) / (np.sum(h[right_mask]) + self.lambd)

    def predict(self, X):
        return np.where(X[:, self.feature_idx] <= self.threshold, self.left_val, self.right_val)

class XGBoostScratch:
    def __init__(self, n_estimators=5, lr=0.1, lambd=1.0):
        self.n_estimators = n_estimators
        self.lr = lr
        self.lambd = lambd
        self.trees = []
        self.init_pred = 0.0 # Log-odds iniziale (probabilità 0.5)

    def _sigmoid(self, x):
        return 1 / (1 + np.exp(-x))

    def fit(self, X, y):
        # Iniziamo con log-odds a 0 per tutti i campioni
        log_odds = np.full(len(y), self.init_pred)

        for _ in range(self.n_estimators):
            # 1. Convertiamo log-odds in probabilità
            p = self._sigmoid(log_odds)
            
            # 2. Calcoliamo Gradienti (g) e Hessiani (h) per LogLoss
            g = p - y
            h = p * (1 - p)

            # 3. Addestriamo un albero su g e h
            tree = XGBoostTree(lambd=self.lambd)
            tree.fit(X, g, h)
            
            # 4. Aggiorniamo le predizioni (log-odds)
            log_odds += self.lr * tree.predict(X)
            self.trees.append(tree)

    def predict_proba(self, X):
        log_odds = np.full(X.shape[0], self.init_pred)
        for tree in self.trees:
            log_odds += self.lr * tree.predict(X)
        return self._sigmoid(log_odds)

    def predict(self, X):
        return (self.predict_proba(X) >= 0.5).astype(int)

In [14]:
# Dati Toy (1-20)
X = np.array([1, 3, 5, 7, 9, 11, 13, 15, 17, 19]).reshape(-1, 1)
y = np.array([0, 0, 0, 0, 0, 1, 1, 1, 1, 1])

# Training
xgb = XGBoostScratch(n_estimators=5, lr=0.1, lambd=1.0)
xgb.fit(X, y)

# Risultati
preds = xgb.predict(X)
probs = xgb.predict_proba(X)

print("--- TEST XGBOOST FROM SCRATCH ---")
print(f"Input:      {X.flatten()}")
print(f"Reali:      {y}")
print(f"Predizioni: {preds}")
print(f"Prob:       {np.round(probs, 3)}")
print(f"Accuratezza: {np.mean(preds == y) * 100}%")

--- TEST XGBOOST FROM SCRATCH ---
Input:      [ 1  3  5  7  9 11 13 15 17 19]
Reali:      [0 0 0 0 0 1 1 1 1 1]
Predizioni: [0 0 0 0 0 1 1 1 1 1]
Prob:       [0.377 0.377 0.377 0.377 0.377 0.623 0.623 0.623 0.623 0.623]
Accuratezza: 100.0%
